# DSC LOGIKA UI 2026 - Final Submission Notebook
Ingestion (raw data -> ABT 70 fitur) + modeling (blend LightGBM + CatBoost, 3-seed averaging) -> file submission.

In [1]:
# ======================
# SETUP PATH (Kaggle)
# ======================
import glob, os

_candidates = glob.glob("/kaggle/input/**/train_applications.csv", recursive=True)
assert len(_candidates) > 0, (
    "train_applications.csv tidak ditemukan di /kaggle/input/. "
    "Pastikan dataset competition sudah ditambahkan lewat tombol '+ Add Input' "
    "di panel kanan Kaggle Notebook."
)
RAW_DIR = os.path.dirname(_candidates[0]) + "/"
print(f"RAW_DIR terdeteksi otomatis: {RAW_DIR}")

PROCESSED_DIR = "/kaggle/working/data/processed/"
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("File yang terdeteksi di RAW_DIR:")
for f in sorted(os.listdir(RAW_DIR)):
    print(" -", f)


RAW_DIR terdeteksi otomatis: /kaggle/input/competitions/data-science-competition-logika-ui-2026/
File yang terdeteksi di RAW_DIR:
 - DATA_DICTIONARY.md
 - account_behavior.csv
 - bureau_observations.csv
 - customer_profiles.csv
 - ecommerce_transactions.csv
 - previous_applications.csv
 - repayment_history.csv
 - sample_submission.csv
 - test_applications.csv
 - train_applications.csv


In [2]:
!pip install -q polars catboost

## Bagian 1 - Ingestion & Feature Engineering
Membangun ABT (Analytical Base Table) dari 9 tabel mentah, leakage-safe, 70 fitur final.

In [3]:
import polars as pl
import numpy as np

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)
pl.Config.set_fmt_float("full")

# RAW_DIR, PROCESSED_DIR, OUTPUT_DIR sudah didefinisikan otomatis di cell setup Kaggle di atas

TRAIN_PATH = RAW_DIR + "train_applications.csv"
TEST_PATH = RAW_DIR + "test_applications.csv"

# 1. Load Spine Data
df_train = pl.read_csv(TRAIN_PATH)
df_test = pl.read_csv(TEST_PATH)

# 2. Parsing Tipe Data Waktu to string
def prep_spine(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("application_datetime").str.to_datetime(),
        pl.col("decision_datetime").str.to_datetime()
    ])

df_train = prep_spine(df_train)
df_test = prep_spine(df_test)

# 3. Validasi Ingestion
print(f"\nDimensi Data Latih: {df_train.shape[0]} baris, {df_train.shape[1]} kolom")
print(f"Dimensi Data Uji: {df_test.shape[0]} baris, {df_test.shape[1]} kolom")

# 4. Cek Distribusi Target (Hanya ada di Data Latih)
print("\nDistribusi Kelas Target:")
print(df_train.group_by("target").len())



Dimensi Data Latih: 70000 baris, 10 kolom
Dimensi Data Uji: 30000 baris, 9 kolom

Distribusi Kelas Target:
shape: (2, 2)
┌────────┬───────┐
│ target ┆ len   │
│ ---    ┆ ---   │
│ i64    ┆ u32   │
╞════════╪═══════╡
│ 1      ┆ 9507  │
│ 0      ┆ 60493 │
└────────┴───────┘


In [4]:
PROFILE_PATH = RAW_DIR + "customer_profiles.csv"

# Load data profil
df_profile = pl.read_csv(PROFILE_PATH)

print(f"Dimensi Profil: {df_profile.shape[0]} baris")

# INVESTIGASI KRITIS: Apakah ada duplikasi customer_id?
duplikasi = df_profile.shape[0] - df_profile.select("customer_id").n_unique()

if duplikasi == 0:
    print("Validasi Sukses: Relasi 1-to-1 terkonfirmasi. Tidak ada duplikat customer_id.")
    df_train = df_train.join(df_profile, on="customer_id", how="left")
    df_test = df_test.join(df_profile, on="customer_id", how="left")
    print(f"Dimensi Train setelah join: {df_train.shape}")
else:
    print(f"BAHAYA TERTANGKAP: Ditemukan {duplikasi} customer_id duplikat!")
    print("Kita harus melakukan deduplikasi sebelum Join!")


Dimensi Profil: 100000 baris
Validasi Sukses: Relasi 1-to-1 terkonfirmasi. Tidak ada duplikat customer_id.
Dimensi Train setelah join: (70000, 17)


In [5]:
# ==========================================
# 1. FRANKENSTEIN: INTRA-SPINE & TIME FEATURES
# ==========================================
import polars as pl

def engineer_spine_features(df: pl.DataFrame) -> pl.DataFrame:
    if df.schema["application_datetime"] in [pl.String, pl.Utf8]:
        df = df.with_columns(pl.col("application_datetime").str.to_datetime())
    if df.schema["decision_datetime"] in [pl.String, pl.Utf8]:
        df = df.with_columns(pl.col("decision_datetime").str.to_datetime())

    is_declared_income_valid = pl.col("declared_monthly_income") > 0
    is_loan_valid = pl.col("loan_amount") > 0

    return df.with_columns([
        ((pl.col("decision_datetime") - pl.col("application_datetime")).dt.total_milliseconds() / (1000 * 60 * 60)).alias("application_to_decision_hours"),
        pl.when(is_loan_valid).then(pl.col("proposed_monthly_installment") / pl.col("loan_amount")).otherwise(None).alias("installment_to_loan_ratio"),
        pl.when(is_declared_income_valid).then(pl.col("proposed_monthly_installment") / pl.col("declared_monthly_income")).otherwise(None).alias("dti_ratio"),
        pl.when(is_declared_income_valid).then(pl.col("loan_amount") / pl.col("declared_monthly_income")).otherwise(None).alias("loan_to_income_ratio")
    ])

print("Memproses Spine Features (Data Latih & Uji)...")
df_train = engineer_spine_features(df_train)
df_test = engineer_spine_features(df_test)
print("Selesai. Dimensi saat ini - Train:", df_train.shape, "| Test:", df_test.shape)


Memproses Spine Features (Data Latih & Uji)...
Selesai. Dimensi saat ini - Train: (70000, 21) | Test: (30000, 20)


In [6]:
# ==========================================
# 2. and 3. FRANKENSTEIN: ACCOUNT BEHAVIOR & BALANCE TREND
# ==========================================
ACC_PATH = RAW_DIR + "account_behavior.csv"

df_acc_lazy = pl.scan_csv(ACC_PATH).with_columns(
    pl.col("month").str.to_date("%Y-%m-%d")
)

def extract_account_features(spine_df: pl.DataFrame, acc_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime", "declared_monthly_income"]).lazy()
    joined = acc_lazy.join(spine_refs, on="customer_id", how="inner")

    valid_behavior = joined.filter(
        pl.col("month").dt.offset_by("1mo") <= pl.col("decision_datetime").dt.date()
    )
    valid_behavior = valid_behavior.sort(["customer_id", "month"])

    features = valid_behavior.group_by("customer_id").agg([
        pl.col("income_like_credits").mean().alias("acc_mean_income_credits"),
        pl.col("low_balance_days").sum().alias("acc_total_low_balance_days"),
        pl.col("outflows").std().alias("acc_std_outflows"),
        pl.col("closing_balance").first().alias("acct_first_closing_balance"),
        pl.col("closing_balance").last().alias("acct_latest_closing_balance")
    ])

    features = features.join(spine_refs.select(["customer_id", "declared_monthly_income"]).unique(), on="customer_id", how="left")
    features = features.with_columns([
        (pl.col("acct_latest_closing_balance") - pl.col("acct_first_closing_balance")).alias("acct_balance_trend"),
        (pl.col("acc_mean_income_credits") - pl.col("declared_monthly_income")).alias("income_discrepancy")
    ])
    features = features.drop(["acct_first_closing_balance", "acct_latest_closing_balance", "declared_monthly_income"])

    return features.collect()

print("Ekstraksi fitur Account Behavior untuk Data Latih...")
acc_features_train = extract_account_features(df_train, df_acc_lazy)
print("Ekstraksi fitur Account Behavior untuk Data Uji...")
acc_features_test = extract_account_features(df_test, df_acc_lazy)

df_train = df_train.join(acc_features_train, on="customer_id", how="left")
df_test = df_test.join(acc_features_test, on="customer_id", how="left")

print("Selesai. Dimensi Latih saat ini:", df_train.shape)
print("Selesai. Dimensi Uji saat ini:", df_test.shape)


Ekstraksi fitur Account Behavior untuk Data Latih...
Ekstraksi fitur Account Behavior untuk Data Uji...
Selesai. Dimensi Latih saat ini: (70000, 26)
Selesai. Dimensi Uji saat ini: (30000, 25)


In [7]:
# ==========================================
# 4. FRANKENSTEIN: E-COMMERCE (ITEM-LEVEL & ORDER-LEVEL)
# ==========================================
ECOM_PATH = RAW_DIR + "ecommerce_transactions.csv"

parsed_transaction_datetime = pl.col("transaction_datetime").str.to_datetime()
df_ecom_lazy = pl.scan_csv(ECOM_PATH).with_columns(parsed_transaction_datetime)

def extract_ecommerce_features(spine_df: pl.DataFrame, ecom_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["application_id", "customer_id", "decision_datetime"]).lazy()
    joined = ecom_lazy.join(spine_refs, on="customer_id", how="inner")

    isBeforeDecision = pl.col("transaction_datetime") < pl.col("decision_datetime")
    valid_tx = joined.filter(isBeforeDecision)

    item_features = valid_tx.group_by("application_id").agg([
        pl.col("product_category").n_unique().alias("ecom_category_diversity")
    ])

    unique_orders = valid_tx.unique(subset=["application_id", "order_id"], keep="first")
    days_before_decision = (pl.col("decision_datetime") - pl.col("transaction_datetime")).dt.total_days().alias("days_before_decision")
    unique_orders = unique_orders.with_columns(days_before_decision)

    is_order_completed = pl.col("order_status") == "completed"
    is_order_cancelled = pl.col("order_status") == "cancelled"
    is_before_30_days = (pl.col("days_before_decision") <= 30)

    order_features = unique_orders.group_by("application_id").agg([
        pl.col("order_total").filter(is_order_completed).sum().alias("ecom_lifetime_spend"),
        pl.col("order_total").filter((is_order_completed) & is_before_30_days).sum().alias("ecom_spend_last_30d"),
        pl.col("order_status").filter(is_order_cancelled).count().alias("ecom_cancelled_count"),
        (pl.col("payment_method").filter(pl.col("payment_method") == "COD").count() / pl.len()).alias("ecom_cod_ratio")
    ])

    features = order_features.join(item_features, on="application_id", how="left")
    return features.collect()

print("Ekstraksi fitur E-commerce untuk Data Latih...")
ecom_features_train = extract_ecommerce_features(df_train, df_ecom_lazy)
print("Ekstraksi fitur E-commerce untuk Data Uji...")
ecom_features_test = extract_ecommerce_features(df_test, df_ecom_lazy)

df_train = df_train.join(ecom_features_train, on="application_id", how="left")
df_test = df_test.join(ecom_features_test, on="application_id", how="left")

print(f"\nDimensi Train setelah E-commerce: {df_train.shape}")
print(f"Dimensi Test setelah E-commerce: {df_test.shape}")


Ekstraksi fitur E-commerce untuk Data Latih...
Ekstraksi fitur E-commerce untuk Data Uji...

Dimensi Train setelah E-commerce: (70000, 31)
Dimensi Test setelah E-commerce: (30000, 30)


In [8]:
# ==========================================
# 5. EKSTRAKSI REPAYMENT HISTORY (CORE TRUTH)
# ==========================================
REPAY_PATH = RAW_DIR + "repayment_history.csv"

df_repay_lazy = pl.scan_csv(REPAY_PATH).with_columns([
    pl.col("due_date").str.to_datetime(),
    pl.col("last_payment_datetime").str.to_datetime()
])

def extract_repayment_features(spine_df: pl.DataFrame, repay_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = repay_lazy.join(spine_refs, on="customer_id", how="inner")

    is_due_date_payment = (pl.col("due_date") < pl.col("decision_datetime"))
    is_past_paid_early = (pl.col("last_payment_datetime") < pl.col("decision_datetime"))
    valid_repay = joined.filter(is_due_date_payment | is_past_paid_early)

    features = valid_repay.group_by("customer_id").agg([
        pl.col("amount_due").sum().alias("repay_total_due"),
        pl.col("amount_paid").sum().alias("repay_total_paid"),
        (pl.col("amount_paid").sum() / pl.col("amount_due").sum()).alias("repay_payment_ratio"),
        pl.col("days_past_due").filter(pl.col("days_past_due") > 0).count().alias("repay_freq_late"),
        pl.col("max_days_past_due").max().alias("repay_historical_worst_dpd"),
        pl.col("last_payment_datetime").is_null().sum().alias("repay_unpaid_installments_count")
    ])
    return features.collect()

print("Ekstraksi fitur Repayment untuk Data Latih...")
repay_features_train = extract_repayment_features(df_train, df_repay_lazy)
print("Ekstraksi fitur Repayment untuk Data Uji...")
repay_features_test = extract_repayment_features(df_test, df_repay_lazy)

df_train = df_train.join(repay_features_train, on="customer_id", how="left")
df_test = df_test.join(repay_features_test, on="customer_id", how="left")

print(f"\nDimensi Train setelah Repayment History: {df_train.shape}")


Ekstraksi fitur Repayment untuk Data Latih...
Ekstraksi fitur Repayment untuk Data Uji...

Dimensi Train setelah Repayment History: (70000, 37)


In [9]:
# ==========================================
# 6. EKSTRAKSI BUREAU OBSERVATIONS (EXTERNAL TRUTH)
# ==========================================
BUREAU_PATH = RAW_DIR + "bureau_observations.csv"

df_bureau_lazy = pl.scan_csv(BUREAU_PATH).with_columns([
    pl.col("report_datetime").str.to_datetime(),
    pl.col("available_datetime").str.to_datetime()
])

def extract_bureau_features(spine_df: pl.DataFrame, bureau_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["application_id", "decision_datetime"]).lazy()
    joined = bureau_lazy.join(spine_refs, on="application_id", how="inner")

    valid_bureau = joined.filter(pl.col("available_datetime") < pl.col("decision_datetime"))
    valid_bureau = valid_bureau.sort(["customer_id", "available_datetime"])

    features = valid_bureau.group_by("application_id").agg([
        pl.col("score").mean().alias("bureau_mean_score"),
        pl.col("outstanding_balance").sum().alias("bureau_total_outstanding"),
        pl.col("active_accounts").sum().alias("bureau_total_active_accounts"),
        pl.col("score").last().alias("bureau_latest_score"),
        ((pl.col("decision_datetime").first() - pl.col("available_datetime").last()).dt.total_days()).alias("bureau_days_since_last_report"),
        (pl.col("revolving_outstanding_balance").sum() /
         pl.when(pl.col("revolving_credit_limit").sum() > 0).then(pl.col("revolving_credit_limit").sum()).otherwise(None)).alias("bureau_utilization_ratio"),
        pl.col("inquiries_last_6m").sum().alias("bureau_total_inquiries_6m"),
        pl.col("delinquent_accounts_last_12m").sum().alias("bureau_total_delinquent_12m")
    ])
    return features.collect()

print("Ekstraksi fitur Bureau untuk Data Latih...")
bureau_features_train = extract_bureau_features(df_train, df_bureau_lazy)
print("Ekstraksi fitur Bureau untuk Data Uji...")
bureau_features_test = extract_bureau_features(df_test, df_bureau_lazy)

df_train = df_train.join(bureau_features_train, on="application_id", how="left")
df_test = df_test.join(bureau_features_test, on="application_id", how="left")

print(f"\nDimensi Train setelah Bureau Observations: {df_train.shape}")


Ekstraksi fitur Bureau untuk Data Latih...
Ekstraksi fitur Bureau untuk Data Uji...

Dimensi Train setelah Bureau Observations: (70000, 45)


In [10]:
# ==========================================
# 7. EKSTRAKSI PREVIOUS APPLICATIONS (REJECTION DENSITY)
# ==========================================
PREV_PATH = RAW_DIR + "previous_applications.csv"

df_prev_lazy = pl.scan_csv(PREV_PATH).with_columns([
    pl.col("application_datetime").str.to_datetime(),
    pl.col("decision_datetime").str.to_datetime()
])

def extract_previous_features(spine_df: pl.DataFrame, prev_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = prev_lazy.join(spine_refs, on="customer_id", how="inner", suffix="_current")

    valid_prev = joined.filter(pl.col("decision_datetime") < pl.col("decision_datetime_current"))
    valid_prev = valid_prev.sort(["customer_id", "decision_datetime"])

    is_app_rejected = pl.col("decision") == "rejected"
    is_app_approved = pl.col("decision") == "approved"

    features = valid_prev.group_by("customer_id").agg([
        pl.len().alias("prev_total_applications"),
        (pl.col("decision").filter(is_app_rejected).count() / pl.len()).alias("prev_rejection_ratio"),
        (pl.col("decision").filter(is_app_approved).count() / pl.len()).alias("prev_approval_ratio"),
        pl.col("requested_amount").mean().alias("prev_mean_requested_amount"),
        ((pl.col("decision_datetime_current").first() - pl.col("decision_datetime").last()).dt.total_days()).alias("days_since_last_prev_app")
    ])
    return features.collect()

print("Ekstraksi fitur Previous Applications untuk Data Latih...")
prev_features_train = extract_previous_features(df_train, df_prev_lazy)
print("Ekstraksi fitur Previous Applications untuk Data Uji...")
prev_features_test = extract_previous_features(df_test, df_prev_lazy)

df_train = df_train.join(prev_features_train, on="customer_id", how="left")
df_test = df_test.join(prev_features_test, on="customer_id", how="left")

print(f"\nDimensi Train setelah Previous Applications: {df_train.shape}")
print(f"Dimensi Test setelah Previous Applications: {df_test.shape}")


Ekstraksi fitur Previous Applications untuk Data Latih...
Ekstraksi fitur Previous Applications untuk Data Uji...

Dimensi Train setelah Previous Applications: (70000, 50)
Dimensi Test setelah Previous Applications: (30000, 49)


In [11]:
# ==========================================
# 8. FRANKENSTEIN ROUND 2: FITUR BARU (INFORMASI GENUINELY BARU)
# ==========================================

def extract_ecom_top_category_share(spine_df: pl.DataFrame, ecom_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["application_id", "customer_id", "decision_datetime"]).lazy()
    joined = ecom_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(
        (pl.col("transaction_datetime") < pl.col("decision_datetime")) &
        (pl.col("order_status") == "completed")
    )
    cat_spend = valid.group_by(["application_id", "product_category"]).agg(
        pl.col("item_total").sum().alias("cat_spend")
    )
    total_spend = cat_spend.group_by("application_id").agg(pl.col("cat_spend").sum().alias("total"))
    top_cat = cat_spend.group_by("application_id").agg(pl.col("cat_spend").max().alias("top_cat_spend"))
    share = top_cat.join(total_spend, on="application_id")
    share = share.with_columns(
        pl.when(pl.col("total") > 0).then(pl.col("top_cat_spend") / pl.col("total")).otherwise(None).alias("ecom_top_category_share")
    ).select(["application_id", "ecom_top_category_share"])
    return share.collect()

def extract_repay_loan_default_rate(spine_df: pl.DataFrame, repay_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = repay_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(
        (pl.col("due_date") < pl.col("decision_datetime")) |
        (pl.col("last_payment_datetime") < pl.col("decision_datetime"))
    )
    loan_max_dpd = valid.group_by(["customer_id", "loan_id"]).agg(
        pl.col("max_days_past_due").max().alias("max_dpd")
    )
    loan_max_dpd = loan_max_dpd.with_columns((pl.col("max_dpd") > 90).alias("is_bad_loan"))
    loan_stats = loan_max_dpd.group_by("customer_id").agg([
        pl.col("loan_id").n_unique().alias("n_past_loans"),
        pl.col("is_bad_loan").sum().alias("n_bad_loans"),
    ])
    loan_stats = loan_stats.with_columns(
        (pl.col("n_bad_loans") / pl.col("n_past_loans")).alias("repay_loan_default_rate")
    ).select(["customer_id", "repay_loan_default_rate"])
    return loan_stats.collect()

def extract_bureau_score_volatility(spine_df: pl.DataFrame, bureau_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["application_id", "decision_datetime"]).lazy()
    joined = bureau_lazy.join(spine_refs, on="application_id", how="inner")
    valid = joined.filter(pl.col("available_datetime") < pl.col("decision_datetime"))
    vol = valid.group_by("application_id").agg(
        pl.col("score").std().alias("bureau_score_volatility")
    )
    return vol.collect()

print("Ekstraksi fitur Round 2 (Data Latih)...")
ecom_share_train = extract_ecom_top_category_share(df_train, df_ecom_lazy)
loan_default_train = extract_repay_loan_default_rate(df_train, df_repay_lazy)
bureau_vol_train = extract_bureau_score_volatility(df_train, df_bureau_lazy)

print("Ekstraksi fitur Round 2 (Data Uji)...")
ecom_share_test = extract_ecom_top_category_share(df_test, df_ecom_lazy)
loan_default_test = extract_repay_loan_default_rate(df_test, df_repay_lazy)
bureau_vol_test = extract_bureau_score_volatility(df_test, df_bureau_lazy)

df_train = df_train.join(ecom_share_train, on="application_id", how="left")
df_train = df_train.join(loan_default_train, on="customer_id", how="left")
df_train = df_train.join(bureau_vol_train, on="application_id", how="left")

df_test = df_test.join(ecom_share_test, on="application_id", how="left")
df_test = df_test.join(loan_default_test, on="customer_id", how="left")
df_test = df_test.join(bureau_vol_test, on="application_id", how="left")

print(f"\nDimensi Train setelah Round 2: {df_train.shape}")
print(f"Dimensi Test setelah Round 2: {df_test.shape}")


Ekstraksi fitur Round 2 (Data Latih)...
Ekstraksi fitur Round 2 (Data Uji)...

Dimensi Train setelah Round 2: (70000, 53)
Dimensi Test setelah Round 2: (30000, 52)


In [12]:
# ==========================================
# 9. FRANKENSTEIN ROUND 3: REKENSI & NORMALISASI FREKUENSI
# ==========================================
def extract_acc_income_cv(spine_df: pl.DataFrame, acc_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = acc_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(pl.col("month").dt.offset_by("1mo") <= pl.col("decision_datetime").dt.date())

    feats = valid.group_by("customer_id").agg([
        pl.col("income_like_credits").mean().alias("_inc_mean"),
        pl.col("income_like_credits").std().alias("_inc_std"),
    ])
    feats = feats.with_columns(
        pl.when(pl.col("_inc_mean") > 0).then(pl.col("_inc_std") / pl.col("_inc_mean")).otherwise(None).alias("acc_income_cv")
    ).select(["customer_id", "acc_income_cv"])
    return feats.collect()

def extract_ecom_freq_recency(spine_df: pl.DataFrame, ecom_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["application_id", "customer_id", "decision_datetime"]).lazy()
    joined = ecom_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(pl.col("transaction_datetime") < pl.col("decision_datetime"))

    unique_orders = valid.unique(subset=["application_id", "order_id"], keep="first")
    unique_orders = unique_orders.filter(pl.col("order_status") == "completed")

    feats = unique_orders.group_by("application_id").agg([
        pl.col("order_id").n_unique().alias("_n_orders"),
        pl.col("transaction_datetime").min().alias("_first_tx"),
        pl.col("transaction_datetime").max().alias("_last_tx"),
        pl.col("decision_datetime").first().alias("_decision_dt"),
    ])
    feats = feats.with_columns([
        (((pl.col("_decision_dt") - pl.col("_first_tx")).dt.total_days() / 30.0).clip(lower_bound=1.0)).alias("_months_span"),
        ((pl.col("_decision_dt") - pl.col("_last_tx")).dt.total_days()).alias("ecom_days_since_last_order"),
    ])
    feats = feats.with_columns(
        (pl.col("_n_orders") / pl.col("_months_span")).alias("ecom_order_freq_per_month")
    ).select(["application_id", "ecom_order_freq_per_month", "ecom_days_since_last_order"])
    return feats.collect()

def extract_repay_most_recent_dpd(spine_df: pl.DataFrame, repay_lazy: pl.LazyFrame) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = repay_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(
        (pl.col("due_date") < pl.col("decision_datetime")) |
        (pl.col("last_payment_datetime") < pl.col("decision_datetime"))
    )
    feats = (
        valid.sort("due_date")
        .group_by("customer_id")
        .agg(pl.col("days_past_due").last().alias("repay_most_recent_dpd"))
    )
    return feats.collect()

print("Ekstraksi fitur Round 3 (Data Latih)...")
acc_cv_train = extract_acc_income_cv(df_train, df_acc_lazy)
ecom_freq_train = extract_ecom_freq_recency(df_train, df_ecom_lazy)
repay_recent_train = extract_repay_most_recent_dpd(df_train, df_repay_lazy)

print("Ekstraksi fitur Round 3 (Data Uji)...")
acc_cv_test = extract_acc_income_cv(df_test, df_acc_lazy)
ecom_freq_test = extract_ecom_freq_recency(df_test, df_ecom_lazy)
repay_recent_test = extract_repay_most_recent_dpd(df_test, df_repay_lazy)

df_train = df_train.join(acc_cv_train, on="customer_id", how="left")
df_train = df_train.join(ecom_freq_train, on="application_id", how="left")
df_train = df_train.join(repay_recent_train, on="customer_id", how="left")

df_test = df_test.join(acc_cv_test, on="customer_id", how="left")
df_test = df_test.join(ecom_freq_test, on="application_id", how="left")
df_test = df_test.join(repay_recent_test, on="customer_id", how="left")

print(f"\nDimensi Train setelah Round 3: {df_train.shape}")
print(f"Dimensi Test setelah Round 3: {df_test.shape}")
print("Total fitur sampai Round 3: 52 (ini config yang Kaggle-proven 0.49351)")


Ekstraksi fitur Round 3 (Data Latih)...
Ekstraksi fitur Round 3 (Data Uji)...

Dimensi Train setelah Round 3: (70000, 57)
Dimensi Test setelah Round 3: (30000, 56)
Total fitur sampai Round 3: 52 (ini config yang Kaggle-proven 0.49351)


In [13]:
# ==========================================
# 10. FRANKENSTEIN ROUND 4: MULTI-WINDOW FEATURES (3/6/12 bulan)
# ==========================================
WINDOWS = [(3, "3m"), (6, "6m"), (12, "12m")]

def extract_repay_windowed(spine_df: pl.DataFrame, repay_lazy: pl.LazyFrame, window_months: int, suffix: str) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = repay_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(
        (pl.col("due_date") < pl.col("decision_datetime")) |
        (pl.col("last_payment_datetime") < pl.col("decision_datetime"))
    )
    cutoff = pl.col("decision_datetime").dt.offset_by(f"-{window_months}mo")
    windowed = valid.filter(pl.col("due_date") >= cutoff)
    feats = windowed.group_by("customer_id").agg([
        pl.col("amount_due").sum().alias(f"repay_due_{suffix}"),
        pl.col("amount_paid").sum().alias(f"repay_paid_{suffix}"),
        pl.col("days_past_due").filter(pl.col("days_past_due") > 0).count().alias(f"repay_freq_late_{suffix}"),
        pl.col("max_days_past_due").max().alias(f"repay_worst_dpd_{suffix}"),
    ])
    feats = feats.with_columns(
        pl.when(pl.col(f"repay_due_{suffix}") > 0)
          .then(pl.col(f"repay_paid_{suffix}") / pl.col(f"repay_due_{suffix}"))
          .otherwise(None)
          .alias(f"repay_payment_ratio_{suffix}")
    ).select(["customer_id", f"repay_freq_late_{suffix}", f"repay_worst_dpd_{suffix}", f"repay_payment_ratio_{suffix}"])
    return feats.collect()

def extract_acc_windowed(spine_df: pl.DataFrame, acc_lazy: pl.LazyFrame, window_months: int, suffix: str) -> pl.DataFrame:
    spine_refs = spine_df.select(["customer_id", "decision_datetime"]).lazy()
    joined = acc_lazy.join(spine_refs, on="customer_id", how="inner")
    valid = joined.filter(pl.col("month").dt.offset_by("1mo") <= pl.col("decision_datetime").dt.date())
    cutoff = pl.col("decision_datetime").dt.offset_by(f"-{window_months}mo").dt.date()
    windowed = valid.filter(pl.col("month") >= cutoff)
    feats = windowed.group_by("customer_id").agg([
        pl.col("low_balance_days").sum().alias(f"acc_low_balance_days_{suffix}"),
        pl.col("outflows").mean().alias(f"acc_mean_outflows_{suffix}"),
        pl.col("income_like_credits").mean().alias(f"acc_mean_income_{suffix}"),
    ])
    return feats.collect()

print("Ekstraksi fitur Round 4 / Multi-window (Data Latih)...")
for months, suffix in WINDOWS:
    rf_train = extract_repay_windowed(df_train, df_repay_lazy, months, suffix)
    af_train = extract_acc_windowed(df_train, df_acc_lazy, months, suffix)
    df_train = df_train.join(rf_train, on="customer_id", how="left")
    df_train = df_train.join(af_train, on="customer_id", how="left")

print("Ekstraksi fitur Round 4 / Multi-window (Data Uji)...")
for months, suffix in WINDOWS:
    rf_test = extract_repay_windowed(df_test, df_repay_lazy, months, suffix)
    af_test = extract_acc_windowed(df_test, df_acc_lazy, months, suffix)
    df_test = df_test.join(rf_test, on="customer_id", how="left")
    df_test = df_test.join(af_test, on="customer_id", how="left")

print(f"\nDimensi Train setelah Round 4: {df_train.shape}")
print(f"Dimensi Test setelah Round 4: {df_test.shape}")
print("Total fitur sekarang: 70 (52 round 1-3 + 18 multi-window)")


Ekstraksi fitur Round 4 / Multi-window (Data Latih)...
Ekstraksi fitur Round 4 / Multi-window (Data Uji)...

Dimensi Train setelah Round 4: (70000, 75)
Dimensi Test setelah Round 4: (30000, 74)
Total fitur sekarang: 70 (52 round 1-3 + 18 multi-window)


In [14]:
# 1. Tarik nama kolom ke dalam himpunan (Set)
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

train_only = train_cols - test_cols
test_only = test_cols - train_cols

print(f"Total Kolom Train: {len(train_cols)}")
print(f"Total Kolom Test: {len(test_cols)}")

print("\nKolom yang HANYA ADA di Data Latih (Seharusnya HANYA 'target'):")
for col in train_only:
    print(f"- {col}")

print("\nKolom yang HANYA ADA di Data Uji (Seharusnya KOSONG):")
for col in test_only:
    print(f"- {col}")


Total Kolom Train: 75
Total Kolom Test: 74

Kolom yang HANYA ADA di Data Latih (Seharusnya HANYA 'target'):
- target

Kolom yang HANYA ADA di Data Uji (Seharusnya KOSONG):


In [15]:
# Dimensi harus sudah benar sebelum menyimpan
assert len(set(df_train.columns) - set(df_test.columns)) == 1, "Masih ada kolom yang tidak sejajar selain target!"

# Buang kolom yang tidak relevan untuk ML (seperti ID pelanggan dan waktu)
# Simpan application_id di test untuk keperluan file submission
drop_cols = ["customer_id", "application_datetime", "decision_datetime"]

df_train_final = df_train.drop(drop_cols)
df_test_final = df_test.drop(drop_cols)

print(f"Dimensi Akhir Train: {df_train_final.shape}")
print(f"Dimensi Akhir Test: {df_test_final.shape}")

# Simpan ke Parquet -- nama file abt4
import os
os.makedirs(PROCESSED_DIR, exist_ok=True)
df_train_final.write_parquet(PROCESSED_DIR + "train_abt4.parquet")
df_test_final.write_parquet(PROCESSED_DIR + "test_abt4.parquet")
print("Data ABT final (70 fitur) berhasil disimpan ke Parquet.")


Dimensi Akhir Train: (70000, 72)
Dimensi Akhir Test: (30000, 71)
Data ABT final (70 fitur) berhasil disimpan ke Parquet.


## Bagian 2 - Modeling
Blend LightGBM (30%) + CatBoost (70%), training 5-fold CV diulang 3 seed berbeda untuk merata-ratakan `test_preds`.

In [16]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

# 1. Memuat ABT final (70 fitur: 52 round 1-3 + 18 multi-window round 4)
df_train = pl.read_parquet(PROCESSED_DIR + "train_abt4.parquet").to_pandas()
df_test = pl.read_parquet(PROCESSED_DIR + "test_abt4.parquet").to_pandas()

# 2. Persiapan Vektor Target dan Matriks Fitur
y = df_train['target']
X = df_train.drop(columns=['application_id', 'target'])

X_test = df_test.drop(columns=['application_id'])
test_ids = df_test['application_id']

# 3. Penanganan Tipe Data Kategorikal
cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"Dimensi Fitur Training: {X.shape}")
print(f"Dimensi Fitur Testing: {X_test.shape}")
print(f"Fitur Kategorikal terdeteksi: {cat_features}")


Dimensi Fitur Training: (70000, 70)
Dimensi Fitur Testing: (30000, 70)
Fitur Kategorikal terdeteksi: ['loan_purpose', 'region', 'employment_type', 'housing_status']


In [ ]:
# ==========================================
# 1. INJEKSI METRIK KUSTOM (KS STATISTIC)
# ==========================================
def lgb_ks_metric(preds, train_data):
    labels = train_data.get_label()
    preds_bad = preds[labels == 1]
    preds_good = preds[labels == 0]
    if len(preds_bad) == 0 or len(preds_good) == 0:
        return 'ks_stat', 0.0, True
    ks_stat = ks_2samp(preds_bad, preds_good).statistic
    return 'ks_stat', float(ks_stat), True

# ==========================================
# 2. PARAMETER (Kaggle-proven, regularisasi manual)
# ==========================================
lgb_params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 15,
    'min_data_in_leaf': 150,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'lambda_l1': 1.0,
    'lambda_l2': 1.0,
    'min_gain_to_split': 0.01,
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1,
    'first_metric_only': True
}

CV_SEEDS = [42, 7, 123]   
W_LGB = 0.3               

cat_idx = [X.columns.get_loc(c) for c in cat_features]

oof_lgb = np.zeros(len(X))   
oof_cat = np.zeros(len(X))

test_preds_lgb_all = []
test_preds_cat_all = []

feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Gain': 0.0})

print("Memulai Training Blend LightGBM + CatBoost (3 CV-seed)...\n")

for cv_seed in CV_SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)
    test_preds_lgb_seed = np.zeros(len(X_test))
    test_preds_cat_seed = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]

        # --- LightGBM ---
        dtrain = lgb.Dataset(X_train_fold, label=y_train_fold, categorical_feature=cat_features, free_raw_data=False)
        dval = lgb.Dataset(X_val_fold, label=y_val_fold, reference=dtrain, categorical_feature=cat_features, free_raw_data=False)
        model_lgb = lgb.train(
            params=lgb_params, train_set=dtrain, num_boost_round=1000,
            valid_sets=[dval], feval=lgb_ks_metric,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)]
        )
        val_lgb = model_lgb.predict(X_val_fold, num_iteration=model_lgb.best_iteration)
        test_preds_lgb_seed += model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration) / skf.n_splits

        # --- CatBoost ---
        X_tr_cat, X_val_cat, X_test_cat = X_train_fold.copy(), X_val_fold.copy(), X_test.copy()
        for c in cat_features:
            X_tr_cat[c] = X_tr_cat[c].astype(str).replace("nan", "missing")
            X_val_cat[c] = X_val_cat[c].astype(str).replace("nan", "missing")
            X_test_cat[c] = X_test_cat[c].astype(str).replace("nan", "missing")

        model_cat = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=5, l2_leaf_reg=3.0,
            random_seed=42, loss_function="Logloss", eval_metric="AUC",
            cat_features=cat_idx, early_stopping_rounds=80, verbose=False,
        )
        model_cat.fit(X_tr_cat, y_train_fold, eval_set=(X_val_cat, y_val_fold), use_best_model=True)
        val_cat = model_cat.predict_proba(X_val_cat)[:, 1]
        test_preds_cat_seed += model_cat.predict_proba(X_test_cat)[:, 1] / skf.n_splits

        if cv_seed == 42:  
            oof_lgb[val_idx] = val_lgb
            oof_cat[val_idx] = val_cat
            feature_importance_df['Gain'] += model_lgb.feature_importance(importance_type='gain') / skf.n_splits
            fold_ks_lgb = ks_2samp(val_lgb[y_val_fold == 1], val_lgb[y_val_fold == 0]).statistic
            fold_ks_cat = ks_2samp(val_cat[y_val_fold == 1], val_cat[y_val_fold == 0]).statistic
            print(f"Fold {fold + 1} | LightGBM KS: {fold_ks_lgb:.4f} | CatBoost KS: {fold_ks_cat:.4f}")

    test_preds_lgb_all.append(test_preds_lgb_seed)
    test_preds_cat_all.append(test_preds_cat_seed)
    print(f"Selesai CV seed={cv_seed}")

# Rata-rata test_preds dari 3 seed, baru di-blend
test_preds_lgb = np.mean(test_preds_lgb_all, axis=0)
test_preds_cat = np.mean(test_preds_cat_all, axis=0)

oof_preds = W_LGB * oof_lgb + (1 - W_LGB) * oof_cat
test_preds = W_LGB * test_preds_lgb + (1 - W_LGB) * test_preds_cat

final_oof_ks = ks_2samp(oof_preds[y == 1], oof_preds[y == 0]).statistic
print(f"\n======================================")
print(f"FINAL OUT-OF-FOLD (OOF) KS SCORE (seed=42): {final_oof_ks:.4f}")
print(f"======================================")

print("\nTop 10 Feature Importance LightGBM (Berdasarkan Information Gain):")
display(feature_importance_df.sort_values(by='Gain', ascending=False).head(10))


Memulai Training Blend LightGBM + CatBoost (3 CV-seed)...

Fold 1 | LightGBM KS: 0.4834 | CatBoost KS: 0.4838
Fold 2 | LightGBM KS: 0.4853 | CatBoost KS: 0.4800
Fold 3 | LightGBM KS: 0.4957 | CatBoost KS: 0.4868
Fold 4 | LightGBM KS: 0.4848 | CatBoost KS: 0.4850
Fold 5 | LightGBM KS: 0.4661 | CatBoost KS: 0.4635
Selesai CV seed=42
Selesai CV seed=7


In [ ]:
# ==========================================
# VISUALISASI KOLMOGOROV-SMIRNOV (KS STATISTIC)
# ==========================================
import matplotlib.pyplot as plt
import numpy as np

def plot_ks_statistic(y_true, y_prob):
    print("Menghitung titik potong KS dan merender grafik...")

    preds_0 = y_prob[y_true == 0]
    preds_1 = y_prob[y_true == 1]

    sort_0 = np.sort(preds_0)
    sort_1 = np.sort(preds_1)

    cdf_0 = np.arange(1, len(sort_0) + 1) / len(sort_0)
    cdf_1 = np.arange(1, len(sort_1) + 1) / len(sort_1)

    ks_stat = 0
    ks_threshold = 0
    cdf_0_val = 0
    cdf_1_val = 0

    all_data_sorted = np.sort(y_prob)
    for x in all_data_sorted:
        cdf_0_temp = np.searchsorted(sort_0, x, side='right') / len(sort_0)
        cdf_1_temp = np.searchsorted(sort_1, x, side='right') / len(sort_1)
        diff = abs(cdf_0_temp - cdf_1_temp)
        if diff > ks_stat:
            ks_stat = diff
            ks_threshold = x
            cdf_0_val = cdf_0_temp
            cdf_1_val = cdf_1_temp

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.hist(preds_0, bins=50, density=True, alpha=0.6, color='dodgerblue', label='Target 0 (Lancar)')
    ax1.hist(preds_1, bins=50, density=True, alpha=0.6, color='crimson', label='Target 1 (Macet)')
    ax1.set_title('Distribusi Probabilitas (OOF Predictions)')
    ax1.set_xlabel('Probabilitas Gagal Bayar')
    ax1.set_ylabel('Kepadatan')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(sort_0, cdf_0, color='dodgerblue', linewidth=2, label='CDF Lancar (0)')
    ax2.plot(sort_1, cdf_1, color='crimson', linewidth=2, label='CDF Macet (1)')
    ax2.axvline(ks_threshold, color='black', linestyle='--', alpha=0.5)
    ax2.plot([ks_threshold, ks_threshold], [cdf_1_val, cdf_0_val], color='limegreen', linewidth=3,
             label=f'KS = {ks_stat:.4f}\n(di Threshold {ks_threshold:.3f})')
    ax2.set_title('Kurva Kumulatif (CDF) & Titik Jarak KS')
    ax2.set_xlabel('Ambang Batas Probabilitas (Threshold)')
    ax2.set_ylabel('Persentase Kumulatif')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_ks_statistic(y, oof_preds)


In [ ]:
# ==========================================
# CETAK FILE SUBMISSION (FINAL)
# ==========================================
import pandas as pd
import datetime
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert len(test_preds) == len(test_ids), "Dimensi prediksi dan ID tidak cocok! Cek arsitektur."

submission = pd.DataFrame({
    'application_id': test_ids,
    'default_probability': test_preds
})

cv_score_str = f"{final_oof_ks:.4f}".replace(".", "")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
filename = f"{OUTPUT_DIR}DSC0135_LearningByDoing_Submission.csv"

submission.to_csv(filename, index=False)
print(f"File submission FINAL berhasil dicetak dan diamankan di: {filename}")

display(submission.head())


In [ ]:
# ==========================================
# SALIN SEBAGAI submission.csv (konvensi umum Kaggle)
# ==========================================
import shutil
shutil.copy(filename, "/kaggle/working/DSC0135_LearningByDoing_Submission.csv")
print("Juga disalin ke /kaggle/working/DSC0135_LearningByDoing_Submission.csv untuk kemudahan submit.")
